# Structure Metrics

![Structure Metrics](https://proto-bio.github.io/proto-assets/images/tool/structure_metrics/hero.png)

This notebook demonstrates `run_structure_metrics`, which computes fast geometric quality metrics on predicted protein structures. We load a bundled example PDB, compute the longest alpha helix length and radius of gyration, and export the results for downstream filtering.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("structure_metrics")
display_overview("structure_metrics")
display_docs_section("structure_metrics", "Background")

## Available tools

In [ ]:
display_available_tools("structure_metrics")

### `run_structure_metrics`

`run_structure_metrics` reads each PDB file with Biotite, assigns secondary structure elements via the P-SEA `annotate_sse()` algorithm (which classifies each residue as helix, sheet, or loop from the C-alpha trace alone using distance and angle patterns), and computes both the longest contiguous alpha helix (in residues) and the radius of gyration (in Angstroms). These two metrics serve as cheap filters to flag common structure-prediction artifacts — runaway single helices, and extended/disordered predictions — before running more expensive downstream analyses.

In [ ]:
from pathlib import Path

from proto_tools import Structure, StructureMetricsConfig, StructureMetricsInput, StructureMetricsOutput, run_structure_metrics
from proto_tools.tools.structure_scoring.structure_metrics.structure_metrics import example_input

In [ ]:
# Display input docs
display_api_reference("structure_metrics", "input", "run_structure_metrics")

# Use the tool's canonical example input — a bundled ESMFold-predicted PDB.
inputs = example_input()

In [ ]:
# Display config docs
display_api_reference("structure_metrics", "config", "run_structure_metrics")

# StructureMetricsConfig has no tool-specific parameters; all defaults are used
config = StructureMetricsConfig()

In [ ]:
# Run the structure metrics computation
result = run_structure_metrics(inputs, config)

In [ ]:
# Display output docs and inspect results
display_api_reference("structure_metrics", "output", "run_structure_metrics")

m = result.metrics[0]
print(f"Longest alpha helix: {m.longest_alpha_helix} residues")
print(f"Radius of gyration:  {m.gyration_radius:.2f} A")

## Filter a Batch of Structures

A common use case is filtering a batch of predicted structures to remove artifacts. The example below applies thresholds calibrated for large globular proteins (~1000–1400 residues) — real helices rarely exceed 50 residues, and well-folded structures of this size typically have a radius of gyration under 45 A.

In [ ]:
LONGEST_ALPHA_THRESHOLD = 50
GYRATION_RADIUS_THRESHOLD = 45.0

# Score a heterogeneous batch: two compact globular folds plus an elongated
# coiled-coil (tropomyosin), which the filter should flag.
batch = [
    ("esmfold_monomer", inputs.structures[0]),
    ("1TIM (TIM barrel)", Structure.from_rcsb("1TIM")),
    ("2TMA (tropomyosin)", Structure.from_rcsb("2TMA")),
]
batch_result = run_structure_metrics(
    StructureMetricsInput(structures=[s for _, s in batch]),
    StructureMetricsConfig(),
)

for (name, _), m in zip(batch, batch_result.metrics):
    reasons = []
    if m.longest_alpha_helix >= LONGEST_ALPHA_THRESHOLD:
        reasons.append(f"long helix ({m.longest_alpha_helix} residues)")
    if m.gyration_radius >= GYRATION_RADIUS_THRESHOLD:
        reasons.append(f"high Rg ({m.gyration_radius:.1f} A)")
    status = "FILTERED: " + ", ".join(reasons) if reasons else "PASS"
    print(f"{name:22} {status}")

## Export Results

Structure metrics can be exported to CSV or JSON for downstream analysis or joining with other per-structure annotations.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("structure_metrics", export_path=output_dir, file_format="csv")
print(f"Exported to {output_dir / 'structure_metrics.csv'}")